# Challenge One: Real-Time Weather Alerts Agent
### Agentic AI with the Google Agent Development Kit (ADK) — Skills Validation Workshop

**Google Cloud Platform (GCP) & Application Default Credentials (ADC) Edition**

**Objectives:**
- Authenticate with Google Cloud using **Application Default Credentials (ADC)**.
- Build an intelligent agent using the **Google Agent Development Kit (ADK)** on **Vertex AI**.
- Use custom tools: **Google Maps Geocoding API** (via ADC Bearer token or API key) and **National Weather Service (NWS)** real-time forecast and alerts.
- Add multi-model support (**Google Gemini 2.0 Flash** and third-party models like **OpenAI GPT-4o** via `LiteLlm`).
- Test and validate the agent across multiple US cities using `reasoning_engines.AdkApp`.

## 1. Install Dependencies

In [1]:
# In Google Cloud Colab Enterprise, install the required packages
!pip install -q --upgrade \
    "google-cloud-aiplatform[agent_engines,adk]" \
    google-adk \
    requests \
    googlemaps \
    litellm

## 2. Google Cloud Authentication & ADC Setup
Initializes Vertex AI and Google Cloud using **Application Default Credentials (ADC)**.

In [2]:
import os
import requests
from typing import Optional, List, Dict, Any
from IPython.display import Markdown, display

import google.auth
from google.auth.transport.requests import Request
import vertexai
from vertexai.preview import reasoning_engines
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm

# 1. Resolve Application Default Credentials (ADC)
try:
    credentials, project_id = google.auth.default(
        scopes=["https://www.googleapis.com/auth/cloud-platform", "https://www.googleapis.com/auth/maps-platform"]
    )
    print(f"[Auth] Successfully loaded ADC for project: {project_id}")
except Exception as e:
    print(f"[Auth Notice] ADC not automatically found ({e}). Ensure 'gcloud auth application-default login' is run.")
    credentials, project_id = None, os.getenv("GOOGLE_CLOUD_PROJECT")

# 2. Configure Region & Vertex AI
PROJECT_ID = project_id or os.getenv("GOOGLE_CLOUD_PROJECT", "qwiklabs-gcp-01-7a38d08f6f78")
LOCATION = os.getenv("GOOGLE_CLOUD_REGION", "us-central1")
MODEL_LOCATION= "global"
env_vars={"GOOGLE_CLOUD_LOCATION": MODEL_LOCATION}

# Instruct Google ADK to use Vertex AI enterprise backend with ADC
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["VERTEXAI_PROJECT"] = PROJECT_ID
os.environ["VERTEXAI_LOCATION"] = LOCATION

if PROJECT_ID != "qwiklabs-gcp-01-7a38d08f6f78":
    vertexai.init(project=PROJECT_ID, location=LOCATION, credentials=credentials)
    print(f"[Vertex AI] Initialized for project '{PROJECT_ID}' in region '{LOCATION}'.")
else:
    print("[Notice] Please set your GCP project ID before initializing Vertex AI.")

[Auth] Successfully loaded ADC for project: qwiklabs-gcp-01-7a38d08f6f78
[Notice] Please set your GCP project ID before initializing Vertex AI.


## 3. Tool 1: Geocoding Function (`get_lat_lon`)
Uses ADC OAuth Bearer tokens or Google Maps API key, with resilient fallbacks for offline testing.

In [3]:
def get_adc_access_token() -> Optional[str]:
    """Fetch a fresh OAuth 2.0 Bearer token from ADC for Google Cloud APIs."""
    try:
        creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform", "https://www.googleapis.com/auth/maps-platform"])
        if not creds.valid:
            creds.refresh(Request())
        return creds.token
    except Exception:
        return None

US_CITY_COORDINATES_FALLBACK = {
    "miami": {"lat": 25.7617, "lon": -80.1918},
    "seattle": {"lat": 47.6062, "lon": -122.3321},
    "chicago": {"lat": 41.8781, "lon": -87.6298},
    "denver": {"lat": 39.7392, "lon": -104.9903},
    "new york": {"lat": 40.7128, "lon": -74.0060},
    "dallas": {"lat": 32.7767, "lon": -96.7970},
}

def get_lat_lon(address: str) -> Optional[Dict[str, float]]:
    """
    Convert a city name, state, or postal address to latitude and longitude coordinates.

    Uses Google Maps Geocoding API authenticated with ADC Bearer token or API key.

    Args:
        address (str): The address or city/state name (e.g., 'Seattle, WA', 'Miami, FL').

    Returns:
        Optional[Dict[str, float]]: Dictionary containing 'lat' and 'lon' floats, or None.
    """
    if not address or not isinstance(address, str):
        return None

    clean_address = address.strip()
    maps_url = "https://maps.googleapis.com/maps/api/geocode/json"

    # 1. Primary: Use ADC Bearer token
    token = get_adc_access_token()
    if token:
        try:
            headers = {"Authorization": f"Bearer {token}"}
            if PROJECT_ID:
                headers["X-Goog-User-Project"] = PROJECT_ID
            resp = requests.get(maps_url, params={"address": clean_address}, headers=headers, timeout=10)
            data = resp.json()
            if data.get("status") == "OK" and data.get("results"):
                loc = data["results"][0]["geometry"]["location"]
                return {"lat": float(loc["lat"]), "lon": float(loc["lng"])}
        except Exception as e:
            print(f"ADC Maps request failed: {e}")

    # 2. Secondary: API Key if set
    api_key = os.getenv("GOOGLE_MAPS_API_KEY")
    if api_key:
        try:
            resp = requests.get(maps_url, params={"address": clean_address, "key": api_key}, timeout=10)
            data = resp.json()
            if data.get("status") == "OK" and data.get("results"):
                loc = data["results"][0]["geometry"]["location"]
                return {"lat": float(loc["lat"]), "lon": float(loc["lng"])}
        except Exception as e:
            print(f"Maps API key error: {e}")

    # 3. Fallbacks (Nominatim & Static dictionary)
    try:
        nom_url = "https://nominatim.openstreetmap.org/search"
        nom_headers = {"User-Agent": "ADK-Weather-Agent-GCP-Workshop/1.0"}
        resp = requests.get(nom_url, params={"q": clean_address, "format": "json", "limit": 1}, headers=nom_headers, timeout=10)
        if resp.status_code == 200 and resp.json():
            return {"lat": float(resp.json()[0]["lat"]), "lon": float(resp.json()[0]["lon"])}
    except Exception:
        pass

    clean_lower = clean_address.lower()
    for city, coords in US_CITY_COORDINATES_FALLBACK.items():
        if city in clean_lower:
            return coords

    return None

## 4. Tool 2: National Weather Service API (`get_extended_weather_forecast` & `get_weather_alerts`)
Retrieves real-time extended forecasts and active severe alerts from the NWS API.

In [4]:
def get_extended_weather_forecast(lat: float, lon: float) -> Optional[List[Dict[str, str]]]:
    """
    Fetch the extended weather forecast from the U.S. National Weather Service API
    based on a given latitude and longitude.

    Args:
        lat (float): Latitude of the location (e.g., 38.8977).
        lon (float): Longitude of the location (e.g., -77.0365).

    Returns:
        Optional[List[Dict[str, str]]]: A list of forecast dictionaries containing period name,
        temperature, wind, and detailed forecast. Returns None if data is unavailable.
    """
    headers = {
        "User-Agent": "(adk-weather-agent-gcp, student-03-29ee00d7ad98@qwiklabs.net)",
        "Accept": "application/geo+json",
    }
    try:
        points_url = f"https://api.weather.gov/points/{lat:.4f},{lon:.4f}"
        resp = requests.get(points_url, headers=headers, timeout=10)
        if resp.status_code != 200:
            return None

        forecast_url = resp.json().get("properties", {}).get("forecast")
        if not forecast_url:
            return None

        forecast_resp = requests.get(forecast_url, headers=headers, timeout=10)
        if forecast_resp.status_code != 200:
            return None

        periods = forecast_resp.json().get("properties", {}).get("periods", [])
        results = []
        for period in periods[:4]:
            results.append({
                "period": str(period.get("name", "Unknown")),
                "temperature": f"{period.get('temperature')} {period.get('temperatureUnit')}",
                "wind": f"{period.get('windSpeed')} {period.get('windDirection')}",
                "summary": str(period.get("shortForecast", "")),
                "detailedForecast": str(period.get("detailedForecast", "")),
            })
        return results
    except Exception as e:
        print(f"NWS forecast error: {e}")
        return None


def get_weather_alerts(lat: float, lon: float) -> Optional[List[Dict[str, str]]]:
    """
    Fetch active weather alerts and severe warnings from the National Weather Service API
    for a specific latitude and longitude coordinate.

    Args:
        lat (float): Latitude of the location.
        lon (float): Longitude of the location.

    Returns:
        Optional[List[Dict[str, str]]]: A list of active alerts with event name, severity,
        urgency, and description.
    """
    headers = {
        "User-Agent": "(adk-weather-agent-gcp, student-03-29ee00d7ad98@qwiklabs.net)",
        "Accept": "application/geo+json",
    }
    try:
        alerts_url = f"https://api.weather.gov/alerts/active?point={lat:.4f},{lon:.4f}"
        resp = requests.get(alerts_url, headers=headers, timeout=10)
        if resp.status_code != 200:
            return []

        features = resp.json().get("features", [])
        alerts = []
        for feature in features:
            props = feature.get("properties", {})
            alerts.append({
                "event": str(props.get("event", "Weather Alert")),
                "severity": str(props.get("severity", "Unknown")),
                "urgency": str(props.get("urgency", "Unknown")),
                "headline": str(props.get("headline", "")),
                "instruction": str(props.get("instruction", "Stay tuned to local news.")),
            })
        return alerts
    except Exception as e:
        print(f"NWS alerts error: {e}")
        return []

## 5. Define Weather Agent Instructions & Gemini Agent (Vertex AI)

In [5]:
WEATHER_AGENT_INSTRUCTIONS = """You are "Pat", an authoritative and friendly real-time Weather and Weather Alerts assistant.
Your goal is to provide accurate, up-to-date weather summaries and severe weather alerts for locations in the United States.

When a user asks for weather or alerts for a city or region:
1. Always call `get_lat_lon` with the user's location to obtain the exact latitude and longitude.
2. If coordinates are found, use them to call:
   - `get_extended_weather_forecast` to get current temperatures, winds, and forecast periods.
   - `get_weather_alerts` to check for active severe weather warnings, watches, or advisories.
3. Structure your response clearly:
   - **Location**: Name of the city/region identified.
   - **Current & Upcoming Forecast**: Temperatures, wind conditions, and short/detailed overview.
   - **Weather Alerts & Safety**: Any active hazards or emergency advisories. If there are none, clearly state that there are no active weather alerts.
4. If the location is outside the United States, politely inform the user that the U.S. National Weather Service API only covers United States locations and territories.
5. If the tools return no data or an error occurs, provide a helpful and courteous explanation.
"""

# Create Gemini Agent with Vertex AI
weather_agent_gemini = Agent(
    name="Pat",
    model="gemini-2.5-flash",
    description="Pat the Friendly Weather and Real-Time Alerts Agent running on Google Cloud.",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[get_lat_lon, get_extended_weather_forecast, get_weather_alerts]
)

print(f"Gemini Agent '{weather_agent_gemini.name}' created on Vertex AI with tools: {[t.__name__ for t in weather_agent_gemini.tools]}")

Gemini Agent 'Pat' created on Vertex AI with tools: ['get_lat_lon', 'get_extended_weather_forecast', 'get_weather_alerts']


## 6. Multi-Model Support: Third-Party Model via LiteLLM (Requirement 6)

In [6]:
# Create Third-Party Model Agent (e.g., OpenAI GPT-4o or Claude 3.5 Sonnet)
weather_agent_gpt = Agent(
    name="Pat_GPT",
    model=LiteLlm(model="openai/gpt-4o"),
    description="Pat the Friendly Weather Agent powered by OpenAI GPT-4o.",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[get_lat_lon, get_extended_weather_forecast, get_weather_alerts]
)

print(f"Third-party Agent '{weather_agent_gpt.name}' created with model: {weather_agent_gpt.model}")

Third-party Agent 'Pat_GPT' created with model: model='openai/gpt-4o' llm_client=<google.adk.models.lite_llm.LiteLLMClient object at 0x7a3803cbfb30>


## 7. Host Agent in `AdkApp` & Session Creation (Vertex AI Reasoning Engines)

In [7]:
app = reasoning_engines.AdkApp(
    agent=weather_agent_gemini,
    enable_tracing=False,
)

user_id = "student-03-29ee00d7ad98@qwiklabs.net"
session = app.create_session(user_id=user_id)
print(f"Created session with ID: {session['id']}")

This legacy setting overrides the new Cloud Console toggle and environment variable controls.
Impact: The Cloud Console may incorrectly show telemetry as 'On' when it is actually 'Off', and the UI toggle will not work.
Action: To fix this and control telemetry, please remove the 'enable_tracing' parameter from your deployment code.
You can then use the 'GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY' environment variable:
agent_engines.create(
  env_vars={
    "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": true|false
  }
)
or the toggle in the Cloud Console: https://console.cloud.google.com/vertex-ai/agents.


Created session with ID: 7089dcdf-a349-4d22-ba67-af9f304cf793


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:1138: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


## 8. Query Function & Multi-City Validation (Requirement 7)

In [8]:
def ask_weather_agent(query: str, current_session_id: str = session['id']):
    """Queries the agent app using stream_query and displays markdown output."""
    print(f"\n" + "=" * 60)
    print(f"USER QUERY: {query}")
    print("=" * 60)

    last_event = None
    for event in app.stream_query(
        user_id=user_id,
        session_id=current_session_id,
        message=query,
    ):
        last_event = event

    if last_event and "content" in last_event and last_event["content"].get("parts"):
        response_text = last_event["content"]["parts"][0].get("text", "")
        display(Markdown(response_text))
    else:
        print("No text response returned.")

# Test across multiple US cities
test_queries = [
    "What is the current weather and are there any active alerts in Seattle, WA?",
    "Give me the real-time weather summary for Miami, Florida.",
    "Is it cold or stormy in Denver, Colorado right now?",
    "What are the weather conditions in Chicago, IL?"
]

for q in test_queries:
    ask_weather_agent(q)


USER QUERY: What is the current weather and are there any active alerts in Seattle, WA?


Here's the weather forecast for Seattle, WA:

**Location**: Seattle, WA

**Current & Upcoming Forecast**:
*   **Today**: A chance of rain. Mostly cloudy. High near 56, with temperatures falling to around 54 in the afternoon. South wind 2 to 12 mph.
*   **Tonight**: Rain. Mostly cloudy. Low around 45, with temperatures rising to around 48 overnight. South southwest wind 9 to 14 mph, with gusts as high as 22 mph.
*   **Wednesday**: Rain. Mostly cloudy. High near 54, with temperatures falling to around 52 in the afternoon. South southwest wind around 9 mph.
*   **Wednesday Night**: Rain. Mostly cloudy, with a low around 42. South southeast wind around 7 mph.

**Weather Alerts & Safety**:
There are no active weather alerts for Seattle, WA at this time.


USER QUERY: Give me the real-time weather summary for Miami, Florida.


Here's the real-time weather summary for Miami, Florida:

**Location**: Miami, Florida

**Current & Upcoming Forecast**:
*   **Today**: Showers and thunderstorms are likely. Mostly sunny, with a high near 82°F. Expect a strong northeast wind around 28 mph, with gusts as high as 37 mph. There's a 60% chance of precipitation, with new rainfall amounts between a quarter and half of an inch possible.
*   **Tonight**: Showers and thunderstorms are likely. Partly cloudy, with a low around 77°F. Northeast wind 14 to 24 mph, with gusts as high as 33 mph. There's a 60% chance of precipitation, with new rainfall amounts between a quarter and half of an inch possible.
*   **Wednesday**: A chance of rain showers before 11 am, then a slight chance of showers and thunderstorms between 11 am and 3 pm, followed by another chance of showers and thunderstorms. Mostly sunny, with a high near 84°F. East wind around 14 mph. There's a 30% chance of precipitation, with new rainfall amounts less than a tenth of an inch possible.
*   **Wednesday Night**: A slight chance of showers and thunderstorms before 8 pm, then a chance of showers and thunderstorms between 8 pm and 2 am, followed by another slight chance of showers and thunderstorms. Mostly clear, with a low around 77°F. East wind 6 to 12 mph. There's a 30% chance of precipitation.

**Weather Alerts & Safety**:
*   **Wind Advisory** is in effect until October 27 at 8:00 PM EDT. Winds this strong can make driving difficult, especially for high-profile vehicles. Use extra caution.
*   **Coastal Flood Warning** is in effect until October 28 at 8:00 PM EDT. Take necessary actions to protect flood-prone property. If travel is required, do not drive around barricades or through water of unknown depth.
*   **Rip Current Statement** is in effect until October 28 at 8:00 PM EDT. Swim near a lifeguard. If caught in a rip current, relax and float. Don't swim against the current. If able, swim in a direction following the shoreline. If unable to escape, face the shore and call or wave for help.


USER QUERY: Is it cold or stormy in Denver, Colorado right now?


It is neither cold nor stormy in Denver, Colorado right now.

**Current & Upcoming Forecast**:
*   **Today**: Expect areas of fog before 11 am, then mostly sunny conditions. The high will be near 66°F, with temperatures falling to around 62°F in the afternoon. There will be an east wind of 3 to 7 mph.
*   **Tonight**: It will be mostly clear with a low around 40°F, with temperatures rising to around 42°F overnight. A west-southwest wind of 3 to 7 mph is expected.

**Weather Alerts & Safety**:
There are no active weather alerts for Denver, CO at this time.


USER QUERY: What are the weather conditions in Chicago, IL?


Here's the weather forecast for Chicago, IL:

**Location**: Chicago, IL

**Current & Upcoming Forecast**:
*   **Today**: Mostly cloudy, with a high near 65°F. Expect a southeast wind of 5 to 10 mph.
*   **Tonight**: Mostly cloudy, with a low around 55°F. There will be a south-southeast wind of 5 to 10 mph.
*   **Wednesday**: Mostly sunny, with a high near 69°F. A south-southwest wind of 5 to 10 mph is expected.
*   **Wednesday Night**: Partly cloudy, with a low around 60°F. South wind 10 to 15 mph.

**Weather Alerts & Safety**:
There are no active weather alerts for Chicago, IL at this time.